In [1]:
import scanpy as sc
import os

adata = sc.read(r'C:\Users\IOC\OneDrive - FIOCRUZ\Área de Trabalho\analysis\data\adata_harmony.h5ad')
adata = adata.raw.to_adata()

from decontx import decontx
# Rodar decontX
result = decontx(
    adata=adata,
    batch_key='library_id',
    seed=42,
    verbose=True,
    cluster_key="leiden_0_5"
)

Starting DecontX
Processing 34931 cells, 17761 genes
Using 17 clusters from 'leiden_0_5'
Processing 14 batches separately
  Processing batch 'CTH92_ML'...


: 

In [ ]:


adata.obs["decontX_contamination"] = result["contamination"]
adata.layers["decontXcounts"] = sp.csr_matrix(result["decontXcounts"])
adata.X = adata.layers["decontXcounts"].copy()

print(f"Células antes: {adata.n_obs}")
adata = adata[adata.obs["decontX_contamination"] < 0.3].copy()
print(f"Células depois: {adata.n_obs}")

sc.pl.umap(
    adata,
    color="decontX_contamination",
    cmap="viridis",
    vmax=0.5  # opcional: corta outliers
)

adata.write_h5ad(r"C:\Users\IOC\OneDrive - FIOCRUZ\Área de Trabalho\analysis\data\adata_harmony_clean.h5ad")

In [1]:
import os

notebookPath = os.path.dirname(os.path.dirname(os.getcwd()))
output_dir = os.path.join(notebookPath, "data", "new")
pastas = sorted(os.listdir(output_dir))

In [2]:
import scanpy as sc
bdata = sc.read(r'C:\Users\IOC\OneDrive - FIOCRUZ\Área de Trabalho\analysis\data\adata_harmony.h5ad')

In [10]:
bdata.layers["norm"] = bdata.X.copy()

In [11]:
bdata.X = bdata.layers["counts"].copy()

In [3]:
bdata = bdata.raw.to_adata()

In [5]:
bdata

AnnData object with n_obs × n_vars = 34931 × 17761
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'mt_outlier', 'doublet_score', 'predicted_doublet', 'library_id', 'controles', 'classificacao', 'estimulo', 'total_counts_rps', 'log1p_total_counts_rps', 'pct_counts_rps', 'total_counts_rpl', 'log1p_total_counts_rpl', 'pct_counts_rpl', 'leiden', 'leiden_0_1', 'leiden_0_2', 'leiden_0_3', 'leiden_0_4', 'leiden_0_5', 'leiden_0_6', 'leiden_0_7', 'leiden_0_8', 'leiden_0_9', 'leiden_final'
    var: 'gene_ids', 'gene_symbol', 'mt', 'rps', 'rpl', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: '_tmp_0.2_42', '_tmp_0.2_43', '_tmp_0.2_44', '_tmp_0.2_45', '_tmp_0.2_46', '_tmp_0.3_42', '_tmp_0.3

In [1]:
import decontx

ImportError: cannot import name 'decontx' from partially initialized module 'decontx' (most likely due to a circular import) (c:\Users\IOC\OneDrive - FIOCRUZ\Área de Trabalho\analysis\src\scripts\decontx.py)

In [ ]:

# Rodar decontX
result = decontx(
    adata=bdata,
    batch_key='library_id',
    seed=42,
    verbose=True,
    cluster_key="leiden_0_5"
)

Starting DecontX
Processing 34931 cells, 17761 genes
Using 17 clusters from 'leiden_0_5'
Processing 14 batches separately
  Processing batch 'CTH92_ML'...


: 

In [ ]:
import os
import scanpy as sc
import numpy as np
import scipy.sparse as sp

# =============================================================
# 0. Diretório de saída
# =============================================================
output_dir = os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), "data", "filtered_h5ad")
os.makedirs(output_dir, exist_ok=True)

# =============================================================
# 1. Carregar amostras como LISTA
# =============================================================

adata_list = [sc.read_h5ad(os.path.join(output_dir, p)) for p in pastas]

# =============================================================
# 2. Rodar decontX por amostra (correto)
# =============================================================
adata_clean = []

for adata, nome in zip(adata_list, pastas):
    print(f"Processando: {nome}")

    # Garantir counts brutos
    if "counts" not in adata.layers:
        adata.layers["counts"] = adata.X.copy()

    # Rodar decontX
    result = decontx(
        adata=adata,
        batch_key=None,
        seed=42,
        verbose=True
    )

    # Adicionar resultados
    adata.obs["decontX_contamination"] = result["contamination"]
    adata.layers["decontXcounts"] = sp.csr_matrix(result["decontXcounts"])

    # Substituir matriz principal
    adata.X = adata.layers["decontXcounts"].copy()

    # =============================================================
    # 3. Filtrar células
    # =============================================================
    print(f"Células antes: {adata.n_obs}")
    adata = adata[adata.obs["decontX_contamination"] < 0.3].copy()
    print(f"Células depois: {adata.n_obs}")

    adata_clean.append(adata)

    sc.pl.umap(
        adata,
        color="decontX_contamination",
        cmap="viridis",
        vmax=0.5  # opcional: corta outliers
    )